# 1 DATA PREPARATION

**Project:** Irish Milk Supply: Growth, Seasonality and Forecasting

**Author:** Temuulen Bulgan

### Purpose of this notebook
This notebook loads the raw data, checks its quality and turns it into one clean monthly table that the rest of the project uses for analysis, SQL and the Power BI dashboard.

### Data sources
| Dataset | Source | Content |
|---|---|---|
| AKM01 | CSO | Milk intake by creameries and pasteurisers; fat and protein content |
| AKM02 | CSO | Milk sales for human consumption (drinking milk) |
| AKM03 | CSO | Production of butter, cheese and skimmed milk powder |
| 32-10-0113-01 | Statistics Canada | Canadian milk production, used for comparison |

### Contents
1. AKM01: inspection and cleaning
2. AKM02: inspection and cleaning
3. AKM03: inspection and cleaning
4. Statistics Canada: inspection and cleaning
5. Combining into one monthly table
6. Saving the clean data

### Output
This notebook saves the cleaned data to `data/processed/ireland_dairy_monthly.csv` (one row per month, one column per measure).

## Setup

In [81]:
import pandas as pd
import numpy as np

# Show all columns when displaying a DataFrame
pd.set_option("display.max_columns", None)

## 1. AKM01: Milk intake, fat and protein content

**Source:** Central Statistics Office (CSO), table AKM01 "Intake of Cows Milk by Creameries and Pasteurisers" (now titled "Intake of Cows Milk by Milk Processors and Co-Ops"). Downloaded from data.cso.ie in December 2022. Licence: CC BY 4.0.

**Structure according to the source:** monthly figures for three statistics (milk intake, fat content and protein content), each split by Domestic and Import source.

In this section I inspect the raw file before cleaning it.

### 1.1 Inspection

#### Load the file and check its size

In [82]:
# Load the raw file
akm01 = pd.read_csv("../data/raw/AKM01.csv")

# Number of rows and columns
akm01.shape

(3444, 5)

#### First and last rows

In [83]:
# First and last 3 rows: format of the data, and where it starts and ends
display(akm01.head(3))
display(akm01.tail(3))

,Statistic Label,Month,Domestic or Import Source,UNIT,VALUE
0,Intake of Cows Milk by Creameries and Pasteuri...,1975 January,Domestic,Million Litres,55.8
1,Intake of Cows Milk by Creameries and Pasteuri...,1975 January,Import,Million Litres,NaN
2,Intake of Cows Milk by Creameries and Pasteuri...,1975 February,Domestic,Million Litres,84.4


,Statistic Label,Month,Domestic or Import Source,UNIT,VALUE
3441,Protein Content,2022 September,Import,%,NaN
3442,Protein Content,2022 October,Domestic,%,3.97
3443,Protein Content,2022 October,Import,%,NaN


First look: the data is in *long* format, with one value per row. The first rows are milk intake for January and February 1975, with one Domestic and one Import row per month. The last rows are protein content for 2022.

#### Column types and missing counts

In [84]:
# Column names, data types and non-missing counts
akm01.info()

<class 'pandas.DataFrame'>
RangeIndex: 3444 entries, 0 to 3443
Data columns (total 5 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   Statistic Label            3444 non-null   str  
 1   Month                      3444 non-null   str  
 2   Domestic or Import Source  3444 non-null   str  
 3   UNIT                       3444 non-null   str  
 4   VALUE                      1856 non-null   str  
dtypes: str(5)
memory usage: 134.7 KB


All five columns are stored as text (`str`). The first four columns are complete, but `VALUE` has only 1,856 non-missing values out of 3,444, and it is stored as text even though it should contain numbers.

#### Categories in each column

In [85]:
# Number of distinct values in each column
akm01.nunique()

Statistic Label                3
Month                        574
Domestic or Import Source      2
UNIT                           2
VALUE                        912
dtype: int64

In [86]:
# Categories in each descriptive column
print(akm01["Statistic Label"].value_counts(), "\n")
print(akm01["Domestic or Import Source"].value_counts(), "\n")
print(akm01["UNIT"].value_counts())

Statistic Label
Intake of Cows Milk by Creameries and Pasteurisers    1148
Fat Content                                           1148
Protein Content                                       1148
Name: count, dtype: int64 

Domestic or Import Source
Domestic    1722
Import      1722
Name: count, dtype: int64 

UNIT
%                 2296
Million Litres    1148
Name: count, dtype: int64


There are 3 statistics (1,148 rows each) and 2 sources (1,722 rows each). Milk intake is measured in million litres; fat and protein in %. `Month` has 574 distinct values (January 1975 to October 2022).

#### Check that every month is complete

In [87]:
# Number of rows for each month (first 3 shown)
akm01["Month"].value_counts().head(3)

Month
1975 January     6
1975 February    6
1975 March       6
Name: count, dtype: int64

In [88]:
# Check that every month has the same number of rows
akm01["Month"].value_counts().unique()

array([6])

In [89]:
# Check that no statistic, month and source combination appears twice
akm01.duplicated(subset=["Statistic Label", "Month", "Domestic or Import Source"]).sum()

np.int64(0)

Structure: all 574 months appear exactly 6 times (3 statistics × 2 sources). The file is a complete grid with no missing or duplicated rows: 574 × 6 = 3,444 rows.

#### Check the `VALUE` column

`VALUE` should contain numbers, but `.info()` shows it is stored as text. I check it in small steps.

In [90]:
# Count empty cells in VALUE in the original file
akm01["VALUE"].isna().sum()

np.int64(1588)

In [91]:
# Convert VALUE to numbers in a separate variable (akm01 is not changed)
# Anything that can't be converted becomes NaN
vals = pd.to_numeric(akm01["VALUE"], errors="coerce")

# Count missing values after the conversion
vals.isna().sum()

np.int64(1591)

After the conversion there are 3 more missing values than before (1,591 vs 1,588). So 3 cells contained text that could not be converted to a number.

In [92]:
# Show the values that were filled in but are not numbers
akm01.loc[vals.isna() & akm01["VALUE"].notna(), "VALUE"].value_counts()

VALUE
.    3
Name: count, dtype: int64

In [93]:
# Count usable numbers
vals.notna().sum()

np.int64(1853)

Summary of `VALUE`:
- 1,853 cells contain numbers
- 1,588 cells are empty
- 3 cells contain "." (CSO's symbol for "not available")
- Total: 3,444

Only the 3 dots stop `VALUE` from being numeric. They can be treated as missing values. All other filled cells are valid numbers.

#### Where the missing values are

I count the missing values for each statistic and source.

In [94]:
# Count missing values for each statistic and source
akm01[vals.isna()].groupby(["Statistic Label", "Domestic or Import Source"]).size()

Statistic Label                                     Domestic or Import Source
Fat Content                                         Domestic                      60
                                                    Import                       574
Intake of Cows Milk by Creameries and Pasteurisers  Import                       323
Protein Content                                     Domestic                      60
                                                    Import                       574
dtype: int64

Imported milk has no fat or protein data at all (574 missing each, which is every month). Imported milk intake has 323 missing months. Domestic fat and protein have 60 missing months each. Domestic milk intake does not appear in the list, so it has no missing values.

In [95]:
# Convert the Month text to real dates
dates = pd.to_datetime(akm01["Month"], format="%Y %B")
dates.head(3)

0   1975-01-01
1   1975-01-01
2   1975-02-01
Name: Month, dtype: datetime64[us]

In [96]:
# Keep only rows that have a number, and add the dates as a temporary column
with_data = akm01[vals.notna()].assign(date=dates)

# First and last month with data for each statistic and source
with_data.groupby(["Statistic Label", "Domestic or Import Source"])["date"].agg(["min", "max"])

min  \
Statistic Label                                    Domestic or Import Source              
Fat Content                                        Domestic                  1980-01-01   
Intake of Cows Milk by Creameries and Pasteurisers Domestic                  1975-01-01   
                                                   Import                    1998-01-01   
Protein Content                                    Domestic                  1980-01-01   

                                                                                    max  
Statistic Label                                    Domestic or Import Source             
Fat Content                                        Domestic                  2022-10-01  
Intake of Cows Milk by Creameries and Pasteurisers Domestic                  2022-10-01  
                                                   Import                    2019-09-01  
Protein Content                                    Domestic                  2022-10-01

Domestic milk intake covers the full period, January 1975 to October 2022. Domestic fat and protein start in January 1980, which explains their 60 missing months (1975–1979). Imported milk intake is only reported from January 1998 to September 2019. The 323 missing months are mostly outside that period (313 months), plus 10 gaps inside it. Imported fat and protein do not appear because they have no data at all.

#### Summary of findings

AKM01 has a complete structure: 574 months from 1975 to 2022, 6 rows per month, no duplicates. The `VALUE` column is text because of 3 "." entries. The missing values are not errors. They come from data CSO doesn't publish, mainly for imported milk. Domestic milk intake has no gaps.

For cleaning, I will convert `VALUE` and `Month` to the right types, keep only domestic data, and reshape the table to one row per month.

### 1.2 Cleaning

#### Convert `VALUE` to numbers

In [99]:
# Work on a copy so the raw data stays unchanged
akm01_clean = akm01.copy()

# Convert VALUE to numbers. The "." entries become NaN
akm01_clean["VALUE"] = pd.to_numeric(akm01_clean["VALUE"], errors="coerce")

akm01_clean["VALUE"].dtype

dtype('float64')

#### Convert `Month` to dates

In [101]:
# Convert the Month text to real dates in a new column
akm01_clean["date"] = pd.to_datetime(akm01_clean["Month"], format="%Y %B")

# Compare the original text and the new date
akm01_clean[["Month", "date"]].head(3)

,Month,date
0,1975 January,1975-01-01
1,1975 January,1975-01-01
2,1975 February,1975-02-01


#### Keep only domestic data
The import figures have no fat or protein data, and intake is only reported for 1998–2019, so I keep domestic data only.

In [103]:
# Keep only domestic rows
akm01_clean = akm01_clean[akm01_clean["Domestic or Import Source"] == "Domestic"]

# Check the result
print(akm01_clean.shape)
print(akm01_clean["Domestic or Import Source"].unique())

(1722, 6)
<StringArray>
['Domestic']
Length: 1, dtype: str


#### Reshape to one row per month
At the moment each month has 3 rows (intake, fat and protein). I turn the 3 statistics into columns, so each month becomes a single row.

In [106]:
# Turn the statistics into columns: one row per month
akm01_wide = akm01_clean.pivot(index="date", columns="Statistic Label", values="VALUE")

print(akm01_wide.shape)
akm01_wide.head(3)

(574, 3)


Statistic Label,Fat Content,Intake of Cows Milk by Creameries and Pasteurisers,Protein Content
date,,,
1975-01-01,NaN,55.8,NaN
1975-02-01,NaN,84.4,NaN
1975-03-01,NaN,193.3,NaN


#### Rename the columns
The CSO names are long, so I rename them to short names that include the unit.

In [107]:
# Short column names with units
akm01_wide = akm01_wide.rename(columns={
    "Intake of Cows Milk by Creameries and Pasteurisers": "intake_ml",
    "Fat Content": "fat_pct",
    "Protein Content": "protein_pct",
})

# Remove the leftover "Statistic Label" name above the columns
akm01_wide.columns.name = None

akm01_wide.head(3)

,fat_pct,intake_ml,protein_pct
date,,,
1975-01-01,NaN,55.8,NaN
1975-02-01,NaN,84.4,NaN
1975-03-01,NaN,193.3,NaN


#### Check the cleaned table

In [108]:
# Put the columns in a logical order
akm01_wide = akm01_wide[["intake_ml", "fat_pct", "protein_pct"]]

# Missing values in each column
akm01_wide.isna().sum()

intake_ml       0
fat_pct        60
protein_pct    60
dtype: int64

In [110]:
# Summary statistics to check the values make sense
akm01_wide.describe()

,intake_ml,fat_pct,protein_pct
count,574.000000,514.000000,514.000000
mean,450.669373,3.828677,3.331732
std,243.622290,0.329923,0.228793
min,55.800000,3.280000,2.860000
25%,217.125000,3.560000,3.180000
50%,456.600000,3.790000,3.280000
75%,637.700000,4.030000,3.470000
max,1181.000000,4.900000,4.040000


All values are in a realistic range. Monthly intake varies from 56 to 1,181 million litres, which reflects the strong seasonal pattern of Irish milk production. Fat is between 3.3% and 4.9% and protein between 2.9% and 4.0%.

#### Cleaning summary

I converted `VALUE` to numbers and `Month` to dates, kept only domestic data, and reshaped the table to one row per month. The result, `akm01_wide`, has 574 rows and 3 columns: `intake_ml`, `fat_pct` and `protein_pct`. Fat and protein are missing for 1975–1979 because CSO starts reporting them in 1980.